# Controlling Randomness with Temperature

## Attribution

Adapted with thanks from **[jaozc/building-with-the-claude-api](https://github.com/jaozc/building-with-the-claude-api/tree/main)**.

Changes for this course: swapped `%pip install` for the **uv** install pattern (uv-managed venvs ship without pip), set the model to **`claude-haiku-4-5`** (the repo default), and rewrote cloud-specific prompts to **Azure**.

### 1. Install dependencies

Install the **`anthropic`** SDK and **`python-dotenv`** with **uv**, pointed at this kernel's interpreter. The step is idempotent, so re-running it no-ops when the packages are already present.

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables from .env file

`load_dotenv()` reads **`ANTHROPIC_API_KEY`** from a local **`.env`** file into the process environment so the SDK can authenticate. Keeping the key out of source is the baseline secrets-hygiene move.

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

### 3. Create an API client

Construct the **`Anthropic`** client (it reads the key from the environment) and pin the **`model`** to **`claude-haiku-4-5`**, the course default.

In [3]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"  # unversioned alias -> current Haiku 4.5 snapshot

### 4. Helper functions

Two small builders append user and assistant turns to a **`messages`** list. The **`chat`** helper adds a **`temperature`** parameter that flows straight through to `client.messages.create()`, so the same conversation can be replayed at different randomness settings.

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

### 5. Compare outputs across temperatures

**Temperature** is a **sampling dial**. At each step the model produces a probability distribution over the next token; temperature reshapes that distribution before a token is drawn. A **lower** value concentrates probability on the single most-likely token, so output is **more focused and repeatable**. A **higher** value flattens the distribution, so **lower-probability tokens** get picked more often and output is **more varied**.

For the **Messages API**, `temperature` is a float that **ranges from `0.0` to `1.0`** and **defaults to `1.0`**.

| `temperature` | Behavior | Good for |
| --- | --- | --- |
| **`0.0`** | Most **focused**, near-deterministic | Extraction, classification, analytical answers |
| **`0.5`** | **Balanced** mix of focus and variety | General assistant replies |
| **`1.0`** | Most **varied and creative** | Brainstorming, story generation |

> Even at `temperature=0.0` the output is **not fully deterministic** - it is the most focused setting, not a guaranteed byte-for-byte repeat.

The cell below sends the **same prompt** three times at `0.0`, `0.5`, and `1.0` so you can read the variance directly.

In [5]:
messages = []

add_user_message(messages, "Generate a one sentence story about a cat.")

response_1 = chat(messages, temperature=0.0)
response_2 = chat(messages, temperature=0.5)
response_3 = chat(messages, temperature=1.0)

print("temperature=0.0:", response_1)
print("temperature=0.5:", response_2)
print("temperature=1.0:", response_3)

temperature=0.0: A curious tabby cat discovered a mysterious door hidden behind the garden shed and disappeared through it, leaving only a glowing paw print behind.
temperature=0.5: A curious tabby cat discovered a mysterious door hidden behind the kitchen cabinets and vanished through it, only to emerge moments later as a wise, ancient sphinx who could finally answer all of her human's questions.
temperature=1.0: A curious tabby cat discovered a hidden door behind the kitchen cabinet and found herself in a magical library where the books whispered their stories aloud.
